# Stage 3: DPO Preference Alignment
## Finance FAQ Assistant — Aligning with Chosen vs. Rejected Responses

This notebook loads the Stage 2 SFT adapter and further aligns it using Direct Preference Optimization (DPO) on 51 chosen/rejected preference pairs, so the model learns to prefer correct, helpful, safe, domain-specific answers over generic, unsafe, or unhelpful ones.

**Runtime:** Google Colab, T4 GPU

In [1]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes
!pip install datasets

## 1. Load the SFT model (Stage 2 output)

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="finance_sft_adapter",  # output of instruction_finetuning.ipynb
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!


RuntimeError: Unsloth: Failed to load model. Both AutoConfig and PeftConfig loading failed.

AutoConfig error: finance_sft_adapter is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

PeftConfig error: Can't find 'adapter_config.json' at 'finance_sft_adapter'



## 2. Load and format the preference dataset

In [ ]:
from datasets import load_dataset

DATA_PATH = "/content/data/preference_dataset.jsonl"  # upload from data/, or clone the repo
pref_ds = load_dataset("json", data_files=DATA_PATH, split="train")
print(pref_ds)
print(pref_ds[0])

In [ ]:
# TRL's DPOTrainer expects columns: prompt, chosen, rejected
# We wrap the prompt in the chat template so it matches how the SFT model was trained.

def format_dpo_example(example):
    prompt_messages = [{"role": "user", "content": example["prompt"]}]
    formatted_prompt = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    return {
        "prompt": formatted_prompt,
        "chosen": example["chosen"] + tokenizer.eos_token,
        "rejected": example["rejected"] + tokenizer.eos_token,
    }

dpo_dataset = pref_ds.map(format_dpo_example)
print(dpo_dataset[0])

## 3. Configure and run DPO training

In [ ]:
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_config = DPOConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    num_train_epochs=3,
    learning_rate=5e-6,         # lower LR than SFT — DPO is sensitive to large updates
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.0,
    lr_scheduler_type="linear",
    seed=42,
    output_dir="outputs_dpo",
    report_to="none",
    beta=0.1,                   # DPO temperature — controls how strongly we penalize the rejected response
    max_length=512,
    max_prompt_length=256,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # Unsloth shares the base weights internally and uses the adapter as the policy; no separate ref model needed
    args=dpo_config,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
)

In [ ]:
dpo_stats = dpo_trainer.train()
print(dpo_stats)

## 4. Save the DPO-aligned model

In [ ]:
model.save_pretrained("finance_dpo_adapter")
tokenizer.save_pretrained("finance_dpo_adapter")
print("DPO-aligned adapter saved to finance_dpo_adapter/")

# Optional: merge LoRA weights into the base model for a standalone deployable model
# merged_model = model.merge_and_unload()
# merged_model.save_pretrained("finance_dpo_merged")

## 5. Test the model after DPO, on the same 10 evaluation questions

In [ ]:
FastLanguageModel.for_inference(model)

EVAL_QUESTIONS = [
    "How can I apply for a personal loan?",
    "What is the difference between a credit card and a debit card?",
    "What happens if I only pay the minimum amount due on my credit card?",
    "What is a SIP?",
    "What is the ideal credit utilization ratio?",
    "What is the difference between fixed and floating interest rates?",
    "What documents do I need to open a savings account?",
    "What happens if I default on a secured loan?",
    "How can I improve my credit score?",
    "What is the difference between TDS and income tax?",
]

def ask_dpo(question, max_new_tokens=120):
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens,
                          do_sample=False, temperature=0.1)
    decoded = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    return decoded.strip()

dpo_answers = {}
for q in EVAL_QUESTIONS:
    ans = ask_dpo(q)
    dpo_answers[q] = ans
    print(f"Q: {q}\nDPO: {ans}\n" + "-"*80)

import json
with open("dpo_model_answers.json", "w") as f:
    json.dump(dpo_answers, f, indent=2)
print("Saved dpo_model_answers.json — used to build reports/final_evaluation.md")

### Observation
Compared to the SFT model, the DPO-aligned model should give answers that are more consistently helpful, complete, and professionally worded — since training explicitly pushed the model away from the weaker/rejected response patterns (vague, unsafe, dismissive) and toward the chosen response patterns (specific, correct, professional).